# 11. Tabellen: Dynamische Programmierung von klein nach groß

In [Notebook 10](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/10_merken.ipynb) hast du Teilprobleme mit einem **Wörterbuch** gemerkt: Die Funktion
ruft sich von oben nach unten selbst auf und rechnet jedes Teilproblem nur einmal. Das hat zwei Schwächen: Die
**Rekursionstiefe** ist begrenzt (bei einem Gitter mit 600 mal 600 Feldern stürzt das Programm ab), und die vielen Funktionsaufrufe
kosten Zeit. In diesem Notebook drehst du die Richtung um: Du füllst eine **Tabelle** (eine Liste oder eine Liste von Listen)
mit **Schleifen von klein nach groß**. Dieselben Teilprobleme, dieselben Formeln, aber keine Rekursion. Das ist die
übliche Form der **dynamischen Programmierung**.

In diesem Notebook lernst du:

- die **Reihenfolge** zu wählen, in der man die Tabelle füllt: Wenn man einen Eintrag berechnet, müssen alle Einträge, die er braucht, schon
  fertig sein,
- aus einer rekursiven Formel aus Notebook 10 eine Tabelle zu machen (Treppenstufen, Gitter, Münzen) und die **beste Lösung
  zurückzuverfolgen**,
- ein **Wort in Wörter zu zerlegen** (`geht[i]`: „die ersten `i` Buchstaben lassen sich zerlegen“),
- **Präfixsummen**: Mit einer Liste „Summe der ersten `i` Zahlen“ rechnest du die Summe über jeden Abschnitt mit **einer** Subtraktion,
- die Größe der Tabelle und die Arbeit je Eintrag zu benutzen, um die **Laufzeit** abzuschätzen,
- typische Fehler zu erkennen (falsche Reihenfolge, Verschiebung um eins).

In der Wettbewerbs-Ecke zerlegst du Wörter in Autokennzeichen, und für Schnelle suchst du mit Präfixsummen die besten Zahlen im
Gewinnspiel „Voll daneben“.

## 1. Treppenstufen als Tabelle

Du kennst die Aufgabe aus Notebook 10: Eine Treppe hat `n` Stufen, du gehst in jedem Schritt 1, 2 oder 3 Stufen nach oben. Die
Formel war `wege(n) = wege(n - 1) + wege(n - 2) + wege(n - 3)` mit `wege(0) = 1`, und `wege(k) = 0` für `k < 0`.

Statt eines Wörterbuchs legen wir jetzt eine **Liste** `tabelle` an, in der `tabelle[k]` die Zahl der Wege zur Stufe `k` ist. Die
Frage ist, **in welcher Reihenfolge** wir sie füllen. Für `tabelle[k]` brauchen wir `tabelle[k - 1]`, `tabelle[k - 2]` und
`tabelle[k - 3]`, also lauter **kleinere** Einträge. Wir füllen deshalb **von klein nach groß**: erst `tabelle[0]`, dann `tabelle[1]`,
`tabelle[2]` und so weiter. Dann ist beim Berechnen von `tabelle[k]` alles Nötige schon fertig.

Eine Falle: Für `k < 3` wären `k - 2` oder `k - 3` negativ, und Python zählt negative Indizes **von hinten** (`tabelle[-1]` ist das
letzte Element), statt einen Fehler zu melden. Prüfe deshalb mit `if k - schritt >= 0`, ob der Eintrag existiert.

#### Aufgabe 1.1

Schreibe `treppe_tabelle(n)`: Sie legt die Liste `tabelle` mit `n + 1` Einträgen an, füllt sie mit einer Schleife von klein nach groß
und gibt das Paar `(tabelle[n], additionen)` zurück. Dabei zählt `additionen`, wie oft man zwei Zahlen addiert hat. Die
Ergebnisse müssen mit Notebook 10 übereinstimmen: 13 Wege bei `n = 5` und 5768 bei `n = 15`.

**Erwartete Ausgabe:**
```
n = 5 : 13 Wege, 6 Tabelleneinträge, 12 Additionen
n = 10 : 274 Wege, 11 Tabelleneinträge, 27 Additionen
n = 15 : 5768 Wege, 16 Tabelleneinträge, 42 Additionen
n = 100 : 180396380815100901214157639 Wege, 101 Tabelleneinträge, 297 Additionen
```

In [ ]:
def treppe_tabelle(n):
    tabelle = [0] * (n + 1)
    tabelle[0] = 1                       # eine Art, schon oben zu sein
    additionen = 0
    # dein Code
    return tabelle[n], additionen

for n in [5, 10, 15, 100]:
    wege, additionen = treppe_tabelle(n)
    print('n =', n, ':', wege, 'Wege,', n + 1, 'Tabelleneinträge,', additionen, 'Additionen')

<details>
<summary>Tipp anzeigen</summary>

`for k in range(1, n + 1):` Für jedes `k` addierst du in einer inneren Schleife über `(1, 2, 3)` die Einträge `tabelle[k - schritt]`,
aber nur, wenn `k - schritt >= 0` gilt. Danach `tabelle[k] = summe`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def treppe_tabelle(n):
    tabelle = [0] * (n + 1)
    tabelle[0] = 1                       # eine Art, schon oben zu sein
    additionen = 0
    for k in range(1, n + 1):            # von klein nach groß
        summe = 0
        for schritt in (1, 2, 3):
            if k - schritt >= 0:         # sonst gibt es den Eintrag nicht (kein negativer Index!)
                summe = summe + tabelle[k - schritt]
                additionen = additionen + 1
        tabelle[k] = summe
    return tabelle[n], additionen

for n in [5, 10, 15, 100]:
    wege, additionen = treppe_tabelle(n)
    print('n =', n, ':', wege, 'Wege,', n + 1, 'Tabelleneinträge,', additionen, 'Additionen')
```

Die Ergebnisse stimmen mit Notebook 10 überein (13 und 5768). Bei `n = 100` ist es dieselbe riesige Zahl wie dort, und das Programm
braucht **keine Rekursion**, also auch keine Grenze der Rekursionstiefe.

</details>

#### Frage 1.2

1. Wie viele Einträge hat die Tabelle für `n = 100`, und wie viel Arbeit macht jeder? Wie groß ist die Laufzeit in O-Notation?
   Vergleiche mit den 301 Aufrufen von `treppe_schnell(100)` aus Notebook 10.
2. Was würde passieren, wenn man die Schleife **von groß nach klein** laufen lässt (`for k in range(n, 0, -1)`)?
3. Welche Einträge der Tabelle braucht man zum Berechnen von `tabelle[k]` überhaupt? Wie viel Speicher bräuchte man mindestens?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. **101** Einträge, für jeden höchstens drei Additionen: Laufzeit **O(n)**. Die Additionen sind die gleichen 297 wie in der Schleife; bei
   Notebook 10 kamen noch die Funktionsaufrufe hinzu (301 Aufrufe für `n = 100`). Es sind **dieselben Teilprobleme** und dieselbe
   Formel, nur die Reihenfolge ist anders: Dort bestimmt die Rekursion die Reihenfolge von oben nach unten, hier wählst du sie selbst
   von unten nach oben. Dafür gibt es **keine Rekursionstiefe** mehr.
2. Dann wäre bei `tabelle[100]` noch nichts anderes berechnet: Die Einträge `tabelle[99]`, `tabelle[98]` und `tabelle[97]` sind
   noch `0` und `tabelle[100]` wird falsch (nämlich `0`). Man muss die Reihenfolge so wählen, dass alles, was ein Eintrag braucht,
   **schon fertig** ist (siehe Abschnitt 4).
3. Nur die **letzten drei** Einträge. Man könnte mit drei Variablen auskommen und die Tabelle weglassen. Meistens (Münzen, Gitter,
   Zurückverfolgen) braucht man aber die ganze Tabelle oder zumindest eine ganze Zeile.

</details>

## 2. Wege im Gitter als Tabelle

Das Gitter aus Notebook 10: Von `(0, 0)` nach `(z, s)` mit Schritten nach unten oder nach rechts, einige Felder sind **Hindernisse**.
Die Formel war `wege(z, s) = wege(z - 1, s) + wege(z, s - 1)`, ein Hindernis hat 0 Wege, das Startfeld 1.

Die Tabelle ist jetzt eine **Liste von Listen**: `tabelle[zeile][spalte]`. Wir legen sie mit `z + 1` Zeilen und `s + 1` Spalten an:
`[[0] * (s + 1) for zeile in range(z + 1)]`. (Nicht `[[0] * (s + 1)] * (z + 1)`: Das würde **dieselbe** Zeile `z + 1`-mal
eintragen, und eine Änderung würde alle Zeilen treffen.)

Ein Feld braucht das Feld **darüber** und das Feld **links davon**. Beide haben eine kleinere Zeilen- oder Spaltennummer. Füllt man
also **Zeile für Zeile, in jeder Zeile von links nach rechts**, dann ist beim Berechnen eines Feldes alles Nötige schon fertig.

#### Aufgabe 2.1

Schreibe `gitter_tabelle(z, s, hindernisse)` mit einer Liste von Listen. Die Funktion gibt die Zahl der Wege zum Feld `(z, s)`
zurück. `hindernisse` ist eine Menge von Tupeln wie `{(1, 1), (2, 3)}`. Die Ergebnisse müssen mit Notebook 10 übereinstimmen
(12870 Wege beim 8×8-Gitter und 51 beim 5×5-Gitter mit drei Hindernissen).

**Erwartete Ausgabe:**
```
8 x 8 , 0 Hindernisse: 12870 Wege, 81 Tabelleneinträge
5 x 5 , 3 Hindernisse: 51 Wege, 36 Tabelleneinträge
600 x 600: 360 Ziffern, die Zahl beginnt mit 39650964 , Tabelleneinträge: 361201
```

In [ ]:
def gitter_tabelle(z, s, hindernisse):
    tabelle = [[0] * (s + 1) for zeile in range(z + 1)]
    # dein Code
    return tabelle[z][s]

def zeige_gitter(z, s, hindernisse):
    wege = gitter_tabelle(z, s, hindernisse)
    print(z, 'x', s, ',', len(hindernisse), 'Hindernisse:', wege, 'Wege,', (z + 1) * (s + 1), 'Tabelleneinträge')

zeige_gitter(8, 8, set())
zeige_gitter(5, 5, {(1, 1), (2, 3), (3, 0)})

# Das große Gitter, bei dem die Rekursion aus Notebook 10 abgestürzt wäre:
gross = gitter_tabelle(600, 600, set())
text = str(gross)
print('600 x 600:', len(text), 'Ziffern, die Zahl beginnt mit', text[:8], ', Tabelleneinträge:', 601 * 601)

<details>
<summary>Tipp anzeigen</summary>

Zwei verschachtelte Schleifen: `for zeile in range(z + 1)` und darin `for spalte in range(s + 1)`. Hindernis: Eintrag 0. Startfeld: 1.
Sonst addierst du `tabelle[zeile - 1][spalte]` (nur wenn `zeile > 0`) und `tabelle[zeile][spalte - 1]` (nur wenn `spalte > 0`).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def gitter_tabelle(z, s, hindernisse):
    tabelle = [[0] * (s + 1) for zeile in range(z + 1)]
    for zeile in range(z + 1):           # Zeile für Zeile
        for spalte in range(s + 1):      # in jeder Zeile von links nach rechts
            if (zeile, spalte) in hindernisse:
                tabelle[zeile][spalte] = 0
            elif zeile == 0 and spalte == 0:
                tabelle[zeile][spalte] = 1
            else:
                summe = 0
                if zeile > 0:
                    summe = summe + tabelle[zeile - 1][spalte]     # von oben
                if spalte > 0:
                    summe = summe + tabelle[zeile][spalte - 1]     # von links
                tabelle[zeile][spalte] = summe
    return tabelle[z][s]

def zeige_gitter(z, s, hindernisse):
    wege = gitter_tabelle(z, s, hindernisse)
    print(z, 'x', s, ',', len(hindernisse), 'Hindernisse:', wege, 'Wege,', (z + 1) * (s + 1), 'Tabelleneinträge')

zeige_gitter(8, 8, set())
zeige_gitter(5, 5, {(1, 1), (2, 3), (3, 0)})

# Das große Gitter, bei dem die Rekursion aus Notebook 10 abgestürzt wäre:
gross = gitter_tabelle(600, 600, set())
text = str(gross)
print('600 x 600:', len(text), 'Ziffern, die Zahl beginnt mit', text[:8], ', Tabelleneinträge:', 601 * 601)
```

Das 600×600-Gitter hat 361 201 Tabelleneinträge. Das Ergebnis ist `math.comb(1200, 600)`, eine Zahl mit 360 Ziffern; weil du
Python-Ganzzahlen benutzt, rechnet Python auch damit exakt. In Notebook 10 hätte `gitter_schnell(600, 600)` mit einem
`RecursionError` aufgegeben. Die Tabelle braucht keine Rekursion, die Schleifen laufen einfach 361 201-mal.

</details>

#### Frage 2.2

1. Wie viele Einträge hat die Tabelle für ein Gitter mit `z + 1` Zeilen und `s + 1` Spalten, wie viel Arbeit macht jeder Eintrag,
   und wie groß ist die Laufzeit? Warum ist das dasselbe wie in Notebook 10?
2. Welche anderen Reihenfolgen der Doppelschleife würden auch funktionieren, welche nicht?
3. Für `tabelle[zeile][spalte]` braucht man nur die eigene Zeile und die **Zeile darüber**. Wie könnte man Speicher sparen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. **(z + 1) · (s + 1)** Einträge (beim 600×600-Gitter 361 201), pro Eintrag höchstens eine Addition: Laufzeit **O(z · s)**. Es sind
   dieselben Teilprobleme `(z, s)` wie bei `gitter_schnell`, nur dass man sie hier **alle** ausrechnet, auch die, die man für das
   Ergebnis nicht braucht (zum Beispiel hinter Hindernissen).
2. **Spalte für Spalte** (außen `spalte`, innen `zeile`) geht auch: Auch dann sind das Feld darüber und das Feld links schon
   fertig. Auch Diagonalen gehen. Falsch wäre jede Reihenfolge, bei der ein Feld **vor** seinem oberen oder linken Nachbarn
   drankommt, zum Beispiel Zeilen von unten nach oben.
3. Nach jeder Zeile braucht man nur noch die letzte fertige Zeile. Zwei Listen der Länge `s + 1` genügen: **O(s)** statt
   O(z · s) Speicher. Das ist bei großen Tabellen wichtig, wenn man nur den Wert (und keinen Weg) braucht.

</details>

## 3. Münzwechsel als Tabelle

Die Münzaufgabe aus Notebook 10: Mit den Münzen `MUENZEN = (1, 4, 6)` soll ein Betrag mit möglichst **wenigen** Münzen gezahlt werden.
Probiere jede Münze als **erste** aus: `wenigste(betrag) = 1 + Minimum von wenigste(betrag - münze)` über alle Münzen, die nicht größer
als der Betrag sind. `wenigste(0) = 0`. Ein Betrag, der gar nicht zu zahlen ist, bekommt `None`.

Die Tabelle `tabelle[b]` enthält die wenigste Münzzahl für den Betrag `b`. Sie hängt von **kleineren** Beträgen ab: Wir füllen sie also
für `b = 1, 2, 3, ...` aufwärts. Ein großer Vorteil: Mit **einer** Tabelle bis zum größten Betrag hat man auch alle kleineren Beträge,
so wie in der Aufgabe unten mit 8, 12, 20 und 25.

Zum Zurückverfolgen merken wir wie in Notebook 10 zusätzlich `wahl[b]`: die erste Münze der besten Möglichkeit. Dort steht `None`,
wenn man den Betrag nicht zahlen kann.

#### Aufgabe 3.1

Schreibe `muenz_tabelle(maximum)`. Sie gibt das Tripel `(tabelle, wahl, pruefungen)` zurück: die Tabelle der wenigsten Münzzahlen für alle
Beträge von 0 bis `maximum`, die Liste `wahl` und die Zahl der geprüften Münzen (`pruefungen`, zähle jede Münze, die nicht größer
als der Betrag ist). Die Ergebnisse müssen mit Notebook 10 übereinstimmen.

**Erwartete Ausgabe:**
```
Betrag 8 : 2 Münzen
Betrag 12 : 2 Münzen
Betrag 20 : 4 Münzen
Betrag 25 : 5 Münzen
26 Tabelleneinträge, 67 geprüfte Münzen
mit 5 und 8: Betrag 11 -> None
mit 5 und 8: Betrag 18 -> 3
mit 5 und 8: Betrag 20 -> 4
```

In [ ]:
MUENZEN = (1, 4, 6)

def muenz_tabelle(maximum):
    tabelle = [None] * (maximum + 1)
    wahl = [None] * (maximum + 1)
    tabelle[0] = 0                       # für den Betrag 0 braucht man keine Münze
    pruefungen = 0
    # dein Code
    return tabelle, wahl, pruefungen

tabelle, wahl, pruefungen = muenz_tabelle(25)
for betrag in [8, 12, 20, 25]:
    print('Betrag', betrag, ':', tabelle[betrag], 'Münzen')
print(len(tabelle), 'Tabelleneinträge,', pruefungen, 'geprüfte Münzen')

MUENZEN = (5, 8)
tabelle, wahl, pruefungen = muenz_tabelle(25)
for betrag in [11, 18, 20]:
    print('mit 5 und 8: Betrag', betrag, '->', tabelle[betrag])

<details>
<summary>Tipp anzeigen</summary>

`for betrag in range(1, maximum + 1)`, darin `for muenze in MUENZEN`. Wenn `muenze <= betrag` ist, sieh in `tabelle[betrag - muenze]` nach.
Ist das nicht `None`, dann ist `rest + 1` eine Möglichkeit für `betrag`. Übernimm sie, wenn `tabelle[betrag]` noch `None` ist oder die neue
Zahl kleiner ist.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
MUENZEN = (1, 4, 6)

def muenz_tabelle(maximum):
    tabelle = [None] * (maximum + 1)
    wahl = [None] * (maximum + 1)
    tabelle[0] = 0                       # für den Betrag 0 braucht man keine Münze
    pruefungen = 0
    for betrag in range(1, maximum + 1):          # von klein nach groß
        for muenze in MUENZEN:
            if muenze <= betrag:
                pruefungen = pruefungen + 1
                rest = tabelle[betrag - muenze]   # ist schon fertig, der Betrag ist kleiner
                if rest is not None:              # None heißt: der Rest geht nicht
                    if tabelle[betrag] is None or rest + 1 < tabelle[betrag]:
                        tabelle[betrag] = rest + 1
                        wahl[betrag] = muenze     # diese Münze ist bis jetzt die beste
    return tabelle, wahl, pruefungen

tabelle, wahl, pruefungen = muenz_tabelle(25)
for betrag in [8, 12, 20, 25]:
    print('Betrag', betrag, ':', tabelle[betrag], 'Münzen')
print(len(tabelle), 'Tabelleneinträge,', pruefungen, 'geprüfte Münzen')

MUENZEN = (5, 8)
tabelle, wahl, pruefungen = muenz_tabelle(25)
for betrag in [11, 18, 20]:
    print('mit 5 und 8: Betrag', betrag, '->', tabelle[betrag])
```

Die Werte sind dieselben wie in Notebook 10: 8 braucht 2 Münzen (4 + 4), 12 braucht 2 (6 + 6), 20 braucht 4 und 25 braucht 5. Mit den
Münzen 5 und 8 geht 11 nicht (`None`), 18 braucht drei Münzen (5 + 5 + 8) und 20 vier (4 · 5). Die Tabelle hat 26 Einträge und
jeder Eintrag prüft höchstens 3 Münzen: Laufzeit **O(Betrag · Zahl der Sorten)**. Weil die Tabelle von klein nach groß gefüllt wird,
steht `tabelle[betrag - muenze]` immer schon fest, auch ein `None`.

</details>

#### Aufgabe 3.2

Schreibe `zahle(wahl, betrag)`: Sie gibt die Liste der Münzen zurück, mit denen man den Betrag zahlt. Dazu folgst du den
Einträgen von `wahl` von hinten nach vorn: Nimm die Münze `wahl[betrag]`, ziehe sie ab und sieh für den Rest wieder nach, bis der Betrag
0 ist. Das sind dieselben Pläne wie in Notebook 10.

**Erwartete Ausgabe:**
```
8 [4, 4]
12 [6, 6]
20 [4, 4, 6, 6]
25 [1, 6, 6, 6, 6]
```

In [ ]:
MUENZEN = (1, 4, 6)

def muenz_tabelle(maximum):
    tabelle = [None] * (maximum + 1)
    wahl = [None] * (maximum + 1)
    tabelle[0] = 0
    pruefungen = 0
    for betrag in range(1, maximum + 1):
        for muenze in MUENZEN:
            if muenze <= betrag:
                pruefungen = pruefungen + 1
                rest = tabelle[betrag - muenze]
                if rest is not None:
                    if tabelle[betrag] is None or rest + 1 < tabelle[betrag]:
                        tabelle[betrag] = rest + 1
                        wahl[betrag] = muenze
    return tabelle, wahl, pruefungen

def zahle(wahl, betrag):
    liste = []
    # dein Code
    return liste

MUENZEN = (1, 4, 6)
tabelle, wahl, pruefungen = muenz_tabelle(25)
for betrag in [8, 12, 20, 25]:
    print(betrag, zahle(wahl, betrag))

<details>
<summary>Tipp anzeigen</summary>

`while betrag > 0:` Hänge `wahl[betrag]` an die Liste an und ziehe diese Münze vom Betrag ab. Mit den Münzen (1, 4, 6) ist jeder Betrag zu
zahlen (wegen der 1), also steht nie `None` in `wahl`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
MUENZEN = (1, 4, 6)

def muenz_tabelle(maximum):
    tabelle = [None] * (maximum + 1)
    wahl = [None] * (maximum + 1)
    tabelle[0] = 0
    pruefungen = 0
    for betrag in range(1, maximum + 1):
        for muenze in MUENZEN:
            if muenze <= betrag:
                pruefungen = pruefungen + 1
                rest = tabelle[betrag - muenze]
                if rest is not None:
                    if tabelle[betrag] is None or rest + 1 < tabelle[betrag]:
                        tabelle[betrag] = rest + 1
                        wahl[betrag] = muenze
    return tabelle, wahl, pruefungen

def zahle(wahl, betrag):
    liste = []
    while betrag > 0:
        muenze = wahl[betrag]            # die beste erste Münze für diesen Betrag
        liste.append(muenze)
        betrag = betrag - muenze
    return liste

MUENZEN = (1, 4, 6)
tabelle, wahl, pruefungen = muenz_tabelle(25)
for betrag in [8, 12, 20, 25]:
    print(betrag, zahle(wahl, betrag))
```

Die Pläne sind dieselben wie in Notebook 10: `[4, 4]` für 8, `[6, 6]` für 12, `[4, 4, 6, 6]` für 20 und `[1, 6, 6, 6, 6]` für 25. Du
speicherst je Betrag nur **eine** Entscheidung, den ganzen Plan baust du beim Zurückverfolgen. Bei gleich guten Möglichkeiten
gewinnt die zuerst probierte Münze, weil nur ein **echt kleinerer** Wert sie ersetzt.

</details>

## 4. Finde den Fehler: die Reihenfolge

Der wichtigste Fehler bei Tabellen ist die **falsche Reihenfolge**: Man berechnet einen Eintrag, bevor die Einträge fertig sind, die er
braucht. Python meldet dann **keinen Fehler**, es liest einfach den Anfangswert der Tabelle (hier `None`).

#### Aufgabe 4.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
MUENZEN = (1, 4, 6)
tabelle = [None] * 9
tabelle[0] = 0

for betrag in range(8, 0, -1):           # von groß nach klein
    for muenze in MUENZEN:
        if muenze <= betrag:
            rest = tabelle[betrag - muenze]
            if rest is not None:
                if tabelle[betrag] is None or rest + 1 < tabelle[betrag]:
                    tabelle[betrag] = rest + 1

print(tabelle)

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
[0, 1, None, None, 1, None, 1, None, None]
```

Ausgegeben wird `[0, 1, None, None, 1, None, 1, None, None]`, obwohl der Betrag 8 mit zwei Münzen zu zahlen ist. Die Schleife beginnt bei
`betrag = 8` und sieht in `tabelle[7]`, `tabelle[4]` und `tabelle[2]` nach. Dort steht noch `None`, denn diese Einträge kommen erst
**später** an die Reihe. Also bleibt `tabelle[8]` leer. Richtig berechnet werden nur die Beträge 1, 4 und 6, die ihre Münze
direkt aus `tabelle[0]` lesen; alle anderen lesen einen noch leeren Eintrag. Eine Fehlermeldung gibt es nicht: Python liest
einfach den Anfangswert `None`. Die Regel: **Alles, was ein Eintrag liest, muss vor ihm berechnet worden sein.** Hier also von klein nach groß
(`range(1, 9)`).

</details>

Der zweite typische Fehler sind die **negativen Indizes**, die Python von hinten zählt (siehe Abschnitt 1):

#### Aufgabe 4.2: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def treppe_falsch(n):
    tabelle = [0] * (n + 1)
    tabelle[0] = 1
    for k in range(1, n + 1):
        tabelle[k] = tabelle[k - 1] + tabelle[k - 2] + tabelle[k - 3]   # keine Prüfung auf k - 3 >= 0
    return tabelle[n]

print(treppe_falsch(1))
print(treppe_falsch(2))
print(treppe_falsch(3))
print(treppe_falsch(4))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
2
2
4
7
```

Ausgegeben werden `2`, `2`, `4` und `7`. Richtig wären `1`, `2`, `4` und `7`: Nur `n = 1` ist falsch. Bei `k = 1` ist `tabelle[k - 2]`
gleich `tabelle[-1]`, also das **letzte** Element, und `tabelle[-2]` ist das vorletzte. Bei `n = 1` hat die Tabelle nur zwei Einträge,
`tabelle[-1]` ist `tabelle[1]` (noch 0), aber `tabelle[-2]` ist `tabelle[0]`, also **1**. So wird die 1 doppelt gezählt. Bei
größeren `n` zeigen die negativen Indizes auf Einträge, die noch 0 sind, und der Fehler fällt nicht auf. Solche Fehler sind
gemein: Das Programm läuft, und die meisten Antworten stimmen. Prüfe darum **immer die Randfälle** (kleinste Eingaben) und
frage vor jedem `tabelle[k - ...]`, ob der Index mindestens 0 ist.

</details>

## 5. Ein Wort in Wörter zerlegen

Die zweite Art von Tabelle: ein **Text** wird von links nach rechts abgearbeitet. Gegeben ist eine Liste erlaubter Wörter, zum Beispiel
`auto`, `bahn`, `hof`, `bus`, `haus`, `tuer`, `ei`. Lässt sich das Wort `autobahnhof` aus **aneinandergehängten** erlaubten Wörtern
bilden (`auto` + `bahn` + `hof`)? Und wenn ja, wie?

Das Teilproblem ist: **Lassen sich die ersten `i` Buchstaben zerlegen?** Die Tabelle heißt `geht`, und `geht[i]` ist `True` oder
`False`. Für `i = 0` (das leere Wort) ist es `True`. Für `i > 0` überlegen wir, wo das **letzte** Wort beginnt, an einer Stelle `j`
mit `0 <= j < i`: Es ist `wort[j:i]`. Es geht, wenn `wort[j:i]` ein erlaubtes Wort ist **und** die ersten `j` Buchstaben sich
zerlegen lassen, also `geht[j]` gilt:

`geht[i] = True`, wenn es ein `j < i` gibt mit `geht[j]` und `wort[j:i] in woerter`.

`j` ist kleiner als `i`, wir füllen die Tabelle also wieder von klein nach groß. Zum Zurückverfolgen merken wir uns in `von[i]`
das `j`, bei dem das letzte Wort beginnt.

#### Aufgabe 5.1

Schreibe `geht_tabelle(wort, woerter)`. Sie gibt das Tripel `(geht, von, pruefungen)` zurück: die Liste `geht` mit `len(wort) + 1` Einträgen
(`True` oder `False`), die Liste `von` (zu jedem `i` mit `geht[i]` der Anfang `j` des letzten Wortes) und die Zahl der geprüften
Stücke `wort[j:i]` (zähle jede Prüfung auf `wort[j:i] in woerter`; Stücke, bei denen `geht[j]` falsch ist, brauchst du gar nicht erst zu prüfen). Hat man für `i` ein Wort gefunden, **genügt das**: Mit `break` brichst du die
innere Schleife ab.

**Erwartete Ausgabe:**
```
autobahnhof : True , 12 Tabelleneinträge, 21 geprüfte Stücke
bushaus : True , 8 Tabelleneinträge, 11 geprüfte Stücke
tuerei : True , 7 Tabelleneinträge, 8 geprüfte Stücke
autobahnen : False , 11 Tabelleneinträge, 18 geprüfte Stücke
hofbus : True , 7 Tabelleneinträge, 9 geprüfte Stücke
[True, False, False, False, True, False, False, False, True, False, False, True]
```

In [ ]:
def geht_tabelle(wort, woerter):
    n = len(wort)
    geht = [False] * (n + 1)
    von = [0] * (n + 1)
    geht[0] = True                       # das leere Wort geht immer
    pruefungen = 0
    # dein Code
    return geht, von, pruefungen

WOERTER = {'auto', 'bahn', 'hof', 'bus', 'haus', 'tuer', 'ei'}

for wort in ['autobahnhof', 'bushaus', 'tuerei', 'autobahnen', 'hofbus']:
    geht, von, pruefungen = geht_tabelle(wort, WOERTER)
    print(wort, ':', geht[len(wort)], ',', len(wort) + 1, 'Tabelleneinträge,', pruefungen, 'geprüfte Stücke')
print(geht_tabelle('autobahnhof', WOERTER)[0])

<details>
<summary>Tipp anzeigen</summary>

Doppelschleife: außen `for i in range(1, n + 1)`, innen `for j in range(i)`. Ist `geht[j]` wahr und `wort[j:i] in woerter`, dann
setze `geht[i] = True`, `von[i] = j` und brich die innere Schleife mit `break` ab. Zähle `pruefungen` bei jedem Test auf `in woerter`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def geht_tabelle(wort, woerter):
    n = len(wort)
    geht = [False] * (n + 1)
    von = [0] * (n + 1)
    geht[0] = True                       # das leere Wort geht immer
    pruefungen = 0
    for i in range(1, n + 1):            # von klein nach groß
        for j in range(i):               # das letzte Wort ist wort[j:i]
            if geht[j]:                  # die ersten j Buchstaben müssen sich zerlegen lassen
                pruefungen = pruefungen + 1
                if wort[j:i] in woerter:
                    geht[i] = True
                    von[i] = j
                    break                # ein Weg genügt
    return geht, von, pruefungen

WOERTER = {'auto', 'bahn', 'hof', 'bus', 'haus', 'tuer', 'ei'}

for wort in ['autobahnhof', 'bushaus', 'tuerei', 'autobahnen', 'hofbus']:
    geht, von, pruefungen = geht_tabelle(wort, WOERTER)
    print(wort, ':', geht[len(wort)], ',', len(wort) + 1, 'Tabelleneinträge,', pruefungen, 'geprüfte Stücke')
print(geht_tabelle('autobahnhof', WOERTER)[0])
```

`autobahnhof` geht (`auto` + `bahn` + `hof`), die Tabelle zeigt es: `geht[4]` (nach `auto`), `geht[8]` (nach `bahn`) und `geht[11]` sind
wahr. `bushaus` (`bus` + `haus`), `tuerei` (`tuer` + `ei`) und `hofbus` gehen ebenfalls. `autobahnen` geht nicht: Nach
`autobahn` bleibt `en`, und `en` ist kein Wort. Weil `geht[j]` zuerst geprüft wird, schneidet man nur dort Stücke ab, wo schon eine
Zerlegung endet; deshalb sind es so wenige Prüfungen. Die Tabelle hat für ein Wort mit `n` Buchstaben `n + 1` Einträge.

</details>

#### Aufgabe 5.2

Schreibe `zerlege(wort, woerter)`: Sie gibt die Liste der Wörter zurück, aus denen sich `wort` zusammensetzt, oder `None`, wenn
das nicht geht. Benutze `geht_tabelle` und folge den Einträgen von `von` von hinten nach vorn: Das letzte Wort ist
`wort[von[i]:i]`, danach ist `i = von[i]`, bis `i` gleich 0 ist.

**Erwartete Ausgabe:**
```
autobahnhof -> ['auto', 'bahn', 'hof']
bushaus -> ['bus', 'haus']
tuerei -> ['tuer', 'ei']
autobahnen -> None
hofbus -> ['hof', 'bus']
ab -> None
['ab', 'cd']
```

In [ ]:
def geht_tabelle(wort, woerter):
    n = len(wort)
    geht = [False] * (n + 1)
    von = [0] * (n + 1)
    geht[0] = True
    pruefungen = 0
    for i in range(1, n + 1):
        for j in range(i):
            if geht[j]:
                pruefungen = pruefungen + 1
                if wort[j:i] in woerter:
                    geht[i] = True
                    von[i] = j
                    break
    return geht, von, pruefungen

WOERTER = {'auto', 'bahn', 'hof', 'bus', 'haus', 'tuer', 'ei'}

def zerlege(wort, woerter):
    geht, von, pruefungen = geht_tabelle(wort, woerter)
    # dein Code
    return []

for wort in ['autobahnhof', 'bushaus', 'tuerei', 'autobahnen', 'hofbus', 'ab']:
    print(wort, '->', zerlege(wort, WOERTER))
print(zerlege('abcd', {'ab', 'abc', 'cd'}))

<details>
<summary>Tipp anzeigen</summary>

`i = len(wort)`. Ist `geht[i]` falsch, gib `None` zurück. Sonst: `while i > 0:` hänge `wort[von[i]:i]` an die Liste an und setze
`i = von[i]`. Am Ende mit `liste.reverse()` die Reihenfolge umdrehen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def geht_tabelle(wort, woerter):
    n = len(wort)
    geht = [False] * (n + 1)
    von = [0] * (n + 1)
    geht[0] = True
    pruefungen = 0
    for i in range(1, n + 1):
        for j in range(i):
            if geht[j]:
                pruefungen = pruefungen + 1
                if wort[j:i] in woerter:
                    geht[i] = True
                    von[i] = j
                    break
    return geht, von, pruefungen

WOERTER = {'auto', 'bahn', 'hof', 'bus', 'haus', 'tuer', 'ei'}

def zerlege(wort, woerter):
    geht, von, pruefungen = geht_tabelle(wort, woerter)
    i = len(wort)
    if not geht[i]:
        return None
    liste = []
    while i > 0:
        liste.append(wort[von[i]:i])     # das letzte Wort
        i = von[i]                       # davor liegt der Rest
    liste.reverse()                      # wir haben von hinten nach vorn gesammelt
    return liste

for wort in ['autobahnhof', 'bushaus', 'tuerei', 'autobahnen', 'hofbus', 'ab']:
    print(wort, '->', zerlege(wort, WOERTER))
print(zerlege('abcd', {'ab', 'abc', 'cd'}))
```

`von[i]` ist genau wie `wahl[betrag]` bei den Münzen die **Entscheidung** am Ende; die Zerlegung entsteht erst beim Zurückverfolgen. Die
letzte Zeile zeigt, warum man nicht einfach **gierig** das längste passende Wort nehmen darf: Bei `abcd` mit den Wörtern `ab`, `abc` und `cd`
würde `abc` den Rest `d` übrig lassen und die Zerlegung scheitern. Die Tabelle findet `ab` + `cd`, denn sie prüft **alle**
Anfänge `j`.

</details>

#### Frage 5.3

1. Für ein Wort mit `n` Buchstaben: Wie viele Stücke `wort[j:i]` würde man **höchstens** prüfen, wenn man bei jedem `i` alle `j < i` probiert
   (ohne `break` und ohne Blick auf `geht[j]`)? Wie groß ist das für `autobahnhof` (11 Buchstaben), und was ist die Laufzeit in O-Notation?
2. Kein erlaubtes Wort ist länger als 4 Buchstaben (`auto`, `bahn`, `haus`, `tuer`). Wie lässt sich die Zahl der Prüfungen
   senken? Wie viele wären es höchstens für `autobahnhof`?
3. Die Teilprobleme `geht(i)` wären bei der Rekursion mit Wörterbuch aus Notebook 10 dieselben. Was ändert sich durch die Tabelle?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Bei `i` gibt es `i` Anfänge `j`, zusammen 1 + 2 + ... + n = n · (n + 1) / 2, bei `n = 11` also **66**. Die Laufzeit ist
   **O(n²)**: Tabelleneinträge (n) mal Arbeit je Eintrag (bis zu n Anfänge). Die tatsächlichen Prüfungen in 5.1 sind viel weniger, weil
   `geht[j]` meistens falsch ist und das `break` die Suche beendet.
2. Ein Wort, das bei `j` beginnt und bei `i` endet, hat höchstens 4 Buchstaben: `j` muss nur von `i - 4` bis `i - 1` laufen
   (`range(max(0, i - 4), i)`). Das sind höchstens 1 + 2 + 3 + 4 · 8 = **38** Prüfungen. Allgemein mit der Länge `L` des längsten
   Wortes: **O(n · L)**. Bei einem Text mit 100 000 Buchstaben und `L = 20` sind das etwa 2 Millionen statt 5 Milliarden Prüfungen.
3. Die Reihenfolge wählst du selbst (von links nach rechts), es gibt keine Rekursionstiefe, und die Arbeit ist dieselbe: Zahl der
   Teilprobleme mal Arbeit je Teilproblem. Der Text darf beliebig lang sein. Bei der Rekursion mit Wörterbuch würde ein Text mit 5000
   Buchstaben an der Rekursionsgrenze scheitern.

</details>

## 6. Präfixsummen: Abschnittssummen mit einer Subtraktion

Eine ganz andere Tabelle: Gegeben ist eine Liste von Zahlen, zum Beispiel die Niederschläge an zehn Tagen
`[3, 1, 4, 1, 5, 9, 2, 6, 5, 3]`. Man fragt viele Male: **Wie viel regnete es von Tag `a` bis Tag `b`?** (Positionen ab 0, beide
eingeschlossen.) Wer jedes Mal alle Zahlen von `a` bis `b` addiert, braucht pro Frage bis zu `n` Additionen.

Besser: Man legt **einmal** eine Tabelle `praefix` an, mit

`praefix[i]` = Summe der **ersten `i` Zahlen** (`praefix[0] = 0` für „keine Zahl“).

Das ist wieder eine Tabelle von klein nach groß: `praefix[i] = praefix[i - 1] + zahlen[i - 1]`. Sie hat `n + 1` Einträge. Die Summe der
Zahlen von Position `a` bis `b` ist dann die Summe der ersten `b + 1` Zahlen **minus** die Summe der ersten `a` Zahlen:

`abschnitt(a, b) = praefix[b + 1] - praefix[a]`

Eine Subtraktion statt bis zu `n` Additionen. (Die Python-Funktion `itertools.accumulate` baut so eine Tabelle automatisch; schreibe
sie zum Verstehen einmal selbst.)

#### Aufgabe 6.1

Schreibe `baue_praefix(zahlen)`, sie gibt die Liste `praefix` mit `len(zahlen) + 1` Einträgen zurück, und `abschnitt(praefix, a, b)`, sie
gibt die Summe der Zahlen von Position `a` bis `b` (beide eingeschlossen) zurück.

**Erwartete Ausgabe:**
```
[0, 3, 4, 8, 9, 14, 23, 25, 31, 36, 39]
Tag 2 bis 5: 19
Tag 0 bis 9: 39
Tag 7 bis 7: 6
```

In [ ]:
def baue_praefix(zahlen):
    praefix = [0]
    # dein Code
    return praefix

def abschnitt(praefix, a, b):
    return 0                             # ersetze diese Zeile

ZAHLEN = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]
praefix = baue_praefix(ZAHLEN)
print(praefix)
print('Tag 2 bis 5:', abschnitt(praefix, 2, 5))
print('Tag 0 bis 9:', abschnitt(praefix, 0, 9))
print('Tag 7 bis 7:', abschnitt(praefix, 7, 7))

<details>
<summary>Tipp anzeigen</summary>

Für jede Zahl hängst du `praefix[-1] + zahl` an (`praefix[-1]` ist der letzte Eintrag). Für den Abschnitt: `praefix[b + 1] - praefix[a]`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def baue_praefix(zahlen):
    praefix = [0]                        # praefix[0]: Summe von keiner Zahl
    for zahl in zahlen:
        praefix.append(praefix[-1] + zahl)   # alte Summe plus die nächste Zahl
    return praefix

def abschnitt(praefix, a, b):
    return praefix[b + 1] - praefix[a]   # Summe der ersten b + 1 Zahlen ohne die ersten a

ZAHLEN = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]
praefix = baue_praefix(ZAHLEN)
print(praefix)
print('Tag 2 bis 5:', abschnitt(praefix, 2, 5))
print('Tag 0 bis 9:', abschnitt(praefix, 0, 9))
print('Tag 7 bis 7:', abschnitt(praefix, 7, 7))
```

Die Tabelle ist `[0, 3, 4, 8, 9, 14, 23, 25, 31, 36, 39]`. Tag 2 bis 5 sind `4 + 1 + 5 + 9 = 19`: `praefix[6] - praefix[2] = 23 - 4`. Ganz hinten steht die
Gesamtsumme 39. Bei `a = 0` ist `praefix[0] = 0`, es wird also nichts abgezogen. Deshalb fängt die Tabelle mit der 0 an.

</details>

#### Aufgabe 6.2

Vergleiche den Aufwand an 1000 Zahlen und 2000 Fragen `(a, b)`. Schreibe `vergleiche_langsam(zahlen, abfragen)`: Sie addiert für jede Frage
alle Zahlen von `a` bis `b` und gibt `(summen, additionen)` zurück (Liste der Summen und Zahl aller Additionen). Schreibe
`vergleiche_praefix(zahlen, abfragen)`: Sie baut einmal die Präfixsummen auf und beantwortet damit alle Fragen. Sie gibt
`(summen, (additionen, subtraktionen))` zurück.

**Erwartete Ausgabe:**
```
2000 Fragen
ohne Präfixsummen: 478902 Additionen
mit Präfixsummen: 1000 Additionen und 2000 Subtraktionen
Alle Summen gleich: True
```

In [ ]:
def baue_praefix(zahlen):
    praefix = [0]
    for zahl in zahlen:
        praefix.append(praefix[-1] + zahl)
    return praefix

def vergleiche_langsam(zahlen, abfragen):
    summen = []
    additionen = 0
    # dein Code
    return summen, additionen

def vergleiche_praefix(zahlen, abfragen):
    summen = []
    praefix = baue_praefix(zahlen)
    additionen = len(zahlen)             # beim Aufbau: eine Addition je Zahl
    subtraktionen = 0
    # dein Code
    return summen, (additionen, subtraktionen)

ZAHLEN = [(i * 37) % 101 for i in range(1000)]          # 1000 "Messwerte"
ABFRAGEN = []
for q in range(2000):                                   # 2000 Fragen (a, b)
    a = (q * 13) % 1000
    b = a + (q * 7) % (1000 - a)
    ABFRAGEN.append((a, b))

summen_langsam, additionen = vergleiche_langsam(ZAHLEN, ABFRAGEN)
summen_schnell, rechnungen = vergleiche_praefix(ZAHLEN, ABFRAGEN)
print(len(ABFRAGEN), 'Fragen')
print('ohne Präfixsummen:', additionen, 'Additionen')
print('mit Präfixsummen:', rechnungen[0], 'Additionen und', rechnungen[1], 'Subtraktionen')
print('Alle Summen gleich:', summen_langsam == summen_schnell)

<details>
<summary>Tipp anzeigen</summary>

`for a, b in abfragen:` zerlegt jedes Paar. Langsam: innere Schleife `for i in range(a, b + 1)`, darin addieren und zählen. Mit Präfixsummen:
`summen.append(praefix[b + 1] - praefix[a])` und eine Subtraktion zählen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def baue_praefix(zahlen):
    praefix = [0]
    for zahl in zahlen:
        praefix.append(praefix[-1] + zahl)
    return praefix

def vergleiche_langsam(zahlen, abfragen):
    summen = []
    additionen = 0
    for a, b in abfragen:
        summe = 0
        for i in range(a, b + 1):
            summe = summe + zahlen[i]
            additionen = additionen + 1
        summen.append(summe)
    return summen, additionen

def vergleiche_praefix(zahlen, abfragen):
    summen = []
    praefix = baue_praefix(zahlen)
    additionen = len(zahlen)             # beim Aufbau: eine Addition je Zahl
    subtraktionen = 0
    for a, b in abfragen:
        summen.append(praefix[b + 1] - praefix[a])
        subtraktionen = subtraktionen + 1
    return summen, (additionen, subtraktionen)

ZAHLEN = [(i * 37) % 101 for i in range(1000)]          # 1000 "Messwerte"
ABFRAGEN = []
for q in range(2000):                                   # 2000 Fragen (a, b)
    a = (q * 13) % 1000
    b = a + (q * 7) % (1000 - a)
    ABFRAGEN.append((a, b))

summen_langsam, additionen = vergleiche_langsam(ZAHLEN, ABFRAGEN)
summen_schnell, rechnungen = vergleiche_praefix(ZAHLEN, ABFRAGEN)
print(len(ABFRAGEN), 'Fragen')
print('ohne Präfixsummen:', additionen, 'Additionen')
print('mit Präfixsummen:', rechnungen[0], 'Additionen und', rechnungen[1], 'Subtraktionen')
print('Alle Summen gleich:', summen_langsam == summen_schnell)
```

Beide Verfahren liefern dieselben 2000 Summen. Das einfache Verfahren braucht für diese Fragen fast eine halbe Million Additionen (478 902), die Präfixsummen
nur 1000 Additionen einmal am Anfang und 2000 Subtraktionen.

</details>

#### Frage 6.3

1. Bei einer Liste mit `n` Zahlen und `q` Fragen: Wie viele Additionen braucht das einfache Verfahren **höchstens**, wie viele
   Rechenschritte das mit Präfixsummen? Wie groß ist beides in O-Notation?
2. Wie groß ist der Unterschied bei `n = 100 000` Zahlen und `q = 100 000` Fragen mit durchschnittlich langen Abschnitten (grob die
   Hälfte der Liste)?
3. Wann lohnen sich Präfixsummen **nicht**?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Einfach: jede Frage bis zu `n` Additionen, zusammen höchstens **q · n**, also **O(q · n)**. In unserer Messung waren es 478 902 Additionen
   für 2000 Fragen (im Mittel etwa 239 je Frage). Mit Präfixsummen: `n` Additionen für die Tabelle und eine Subtraktion je Frage, also
   **n + q** Schritte, **O(n + q)**: Die Tabelle ist **einmal** nötig, jede Frage kostet dann **O(1)**.
2. Einfach: etwa 100 000 · 50 000 = **5 Milliarden** Additionen (viel zu langsam im Wettbewerb). Mit Präfixsummen: etwa **200 000** Schritte.
3. Wenn es nur **eine** oder sehr wenige Fragen gibt, denn dann kostet der Aufbau der Tabelle (`n` Additionen) genauso viel wie
   die einfache Rechnung. Außerdem, wenn sich die Zahlen **zwischen** den Fragen ändern: Dann stimmt die Tabelle nicht mehr (man müsste sie neu bauen).

</details>

#### Aufgabe 6.4: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
zahlen = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]

p = []
summe = 0
for zahl in zahlen:
    summe = summe + zahl
    p.append(summe)                      # p[i] = zahlen[0] + ... + zahlen[i]

print(p)
print(p[5] - p[2])                       # Summe von Position 2 bis 5?
print(p[5] - p[0 - 1])                   # Summe von Position 0 bis 5?

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
[3, 4, 8, 9, 14, 23, 25, 31, 36, 39]
15
-16
```

Ausgegeben wird `[3, 4, 8, 9, 14, 23, 25, 31, 36, 39]`, dann `15` und dann `-16`. Richtig wären `19` und `3 + 1 + 4 + 1 + 5 + 9 = 23`. Hier
steht in `p[i]` schon die Zahl an Position `i` mit drin (die Tabelle hat keinen Eintrag für „keine Zahl“). Dann ist `p[5] - p[2]` die
Summe von Position **3** bis 5, die Zahl bei Position `a` fehlt (**Verschiebung um eins**): `15 = 1 + 5 + 9`. Wer das merkt und
`p[a - 1]` nimmt, bekommt für `a = 0` den Eintrag `p[-1]`, und das ist in Python der **letzte** Eintrag (die Gesamtsumme 39): `23 - 39 = -16`.
Die saubere Lösung ist die Tabelle aus 6.1 mit einer `0` am Anfang: `praefix[i]` ist die Summe der **ersten `i`** Zahlen,
der Abschnitt von `a` bis `b` ist `praefix[b + 1] - praefix[a]`, und `a = 0` ist kein Sonderfall. Probiere solche Formeln immer an einem
Abschnitt am **Rand** (Anfang, Ende, nur eine Zahl) aus.

</details>

## Wettbewerbs-Ecke 1: Auto-Scrabble

*Angelehnt an Aufgabe 4 „Auto-Scrabble“ der 1. Runde des 36. Bundeswettbewerbs Informatik (2017/18).*

Kann man ein Wort mit **Autokennzeichen** schreiben? Ein Kennzeichen besteht aus einem **Kürzel** für eine Stadt oder einen Kreis
(1 bis 3 Buchstaben, zum Beispiel `B`, `WI`, `HEI`), einem **Mittelteil** aus ein oder zwei Buchstaben und einer Zahl, die wir
ignorieren. `HEI-KE` schreibt das Wort `HEIKE`, und `B-W` und `IN-F` zusammen schreiben `BWINF`. Das Wort darf aus **mehreren**
Kennzeichen zusammengesetzt werden, aber jedes Kennzeichen muss **vollständig** benutzt werden.

Wir vereinfachen: Alle Wörter bestehen nur aus den Buchstaben `A` bis `Z` (Umlaute sind als `AE`, `OE`, `UE` geschrieben, `ß` als `SS`),
und von den 684 Kürzeln der Liste benutzen wir die 645 ohne Umlaute. Die Datei `daten/kuerzel.txt` enthält je Zeile ein Kürzel, die Datei
`daten/kennzeichen_woerter.txt` je Zeile ein Wort.

Ein Kennzeichen ist also ein Stück des Wortes mit **2 bis 5 Buchstaben**. Die Funktion `ist_kennzeichen(stueck)` ist fertig und gibt den
Text `KÜRZEL-MITTE` zurück (zum Beispiel `'IN-F'`), oder `None`, wenn man das Stück nicht als Kennzeichen schreiben kann. Du brauchst die
Tabelle aus Abschnitt 5: `geht[i]` ist wahr, wenn die ersten `i` Buchstaben mit Kennzeichen geschrieben werden können. Das letzte
Kennzeichen ist das Stück `wort[j:i]` mit `j = i - laenge` für `laenge` von 2 bis 5.

#### Aufgabe E.1

Schreibe `zerlege_kennzeichen(wort)`: Sie gibt die Liste der Kennzeichen (als Texte wie `'B-W'`) zurück, mit denen man das Wort schreiben
kann, oder `None`. Lege dazu die Listen `geht` und `letztes` an (`letztes[i]` ist das letzte Kennzeichen der Zerlegung der ersten `i`
Buchstaben) und verfolge sie zum Schluss von hinten zurück. Ein Kennzeichen mit Text `'IN-F'` verbraucht `len(text) - 1` Buchstaben, denn der
Bindestrich gehört nicht zum Wort. Die Aufteilung kann sich von der in den Lösungshinweisen unterscheiden; wichtig ist, **ob** ein
Wort geht.

**Erwartete Ausgabe:**
```
IN-F None HEI-KE
BIBER -> B-IB E-R
BUNDESWETTBEWERB -> B-UN D-E S-W E-TT B-E W-E R-B
CLINTON -> C-LI NT-ON
ETHERNET -> E-T H-E R-N E-T
INFORMATIK -> IN-FO R-M AT-IK
SOFTWARE -> S-O F-T W-A R-E
TRUMP -> TR-U M-P
HEIKE -> H-EI K-E
TIMO -> nicht möglich
TSCHUESS -> TS-C H-UE S-S
DONAUDAMPFSCH... -> 16 Kennzeichen
RINDFLEISCHET... -> nicht möglich
VERKEHRSWEGEP... -> 19 Kennzeichen
```

In [ ]:
def lies_liste(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        return datei.read().split()

KUERZEL = set(lies_liste('daten/kuerzel.txt'))
KZ_WOERTER = lies_liste('daten/kennzeichen_woerter.txt')

def ist_kennzeichen(stueck):
    """Gibt 'KÜRZEL-MITTE' zurück, wenn das Stück ein Kennzeichen sein kann, sonst None."""
    for k in range(1, 4):                # Länge des Kürzels
        if stueck[:k] in KUERZEL and 1 <= len(stueck) - k <= 2:
            return stueck[:k] + '-' + stueck[k:]
    return None

def zerlege_kennzeichen(wort):
    n = len(wort)
    geht = [False] * (n + 1)
    letztes = [None] * (n + 1)           # letztes[i]: letztes Kennzeichen der Zerlegung der ersten i Buchstaben
    geht[0] = True
    # dein Code
    return None

print(ist_kennzeichen('INF'), ist_kennzeichen('TIMO'), ist_kennzeichen('HEIKE'))
for wort in KZ_WOERTER:
    liste = zerlege_kennzeichen(wort)
    kurz = wort if len(wort) <= 16 else wort[:13] + '...'
    if liste is None:
        print(kurz, '-> nicht möglich')
    elif len(wort) <= 16:
        print(kurz, '->', ' '.join(liste))
    else:
        print(kurz, '->', len(liste), 'Kennzeichen')

<details>
<summary>Tipp anzeigen</summary>

Wie in 5.1, nur dass das letzte Stück `wort[j:i]` mit `j = i - laenge` für `laenge in range(2, 6)` gebildet wird und statt `in woerter` die
Funktion `ist_kennzeichen` fragt (Achtung: `j` darf nicht negativ werden). Merke dir in `letztes[i]` den Text. Zurückverfolgen: `i = n`, dann
`text = letztes[i]`, anhängen, `i = i - (len(text) - 1)`, bis `i` 0 ist.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_liste(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        return datei.read().split()

KUERZEL = set(lies_liste('daten/kuerzel.txt'))
KZ_WOERTER = lies_liste('daten/kennzeichen_woerter.txt')

def ist_kennzeichen(stueck):
    """Gibt 'KÜRZEL-MITTE' zurück, wenn das Stück ein Kennzeichen sein kann, sonst None."""
    for k in range(1, 4):                # Länge des Kürzels
        if stueck[:k] in KUERZEL and 1 <= len(stueck) - k <= 2:
            return stueck[:k] + '-' + stueck[k:]
    return None

def zerlege_kennzeichen(wort):
    n = len(wort)
    geht = [False] * (n + 1)
    letztes = [None] * (n + 1)           # letztes[i]: letztes Kennzeichen der Zerlegung der ersten i Buchstaben
    geht[0] = True
    for i in range(1, n + 1):            # von klein nach groß
        for laenge in range(2, 6):       # ein Kennzeichen hat 2 bis 5 Buchstaben
            j = i - laenge
            if j >= 0 and geht[j] and not geht[i]:
                text = ist_kennzeichen(wort[j:i])
                if text is not None:
                    geht[i] = True
                    letztes[i] = text
    if not geht[n]:
        return None
    liste = []
    i = n
    while i > 0:                         # von hinten nach vorn zurückverfolgen
        text = letztes[i]
        liste.append(text)
        i = i - (len(text) - 1)          # der Bindestrich zählt nicht
    liste.reverse()
    return liste

print(ist_kennzeichen('INF'), ist_kennzeichen('TIMO'), ist_kennzeichen('HEIKE'))
for wort in KZ_WOERTER:
    liste = zerlege_kennzeichen(wort)
    kurz = wort if len(wort) <= 16 else wort[:13] + '...'
    if liste is None:
        print(kurz, '-> nicht möglich')
    elif len(wort) <= 16:
        print(kurz, '->', ' '.join(liste))
    else:
        print(kurz, '->', len(liste), 'Kennzeichen')
```

Alle Wörter außer `TIMO` und dem langen `RINDFLEISCH...` lassen sich schreiben. `TIMO` geht nicht: Kein Kürzel der Liste ist `T`, `TI` oder `TIM`
(es gibt nur `TIR`), also kann kein Kennzeichen am Anfang stehen. Die Aufteilung von `BIBER` und den anderen Wörtern kann sich von der im
Lösungsnotebook unterscheiden, weil wir die Möglichkeiten in anderer Reihenfolge probieren; wichtig ist, **ob** ein Wort geht.

</details>

Ein Wort der Länge `n` kostet bei dieser Tabelle nur etwa `n · 4` Prüfungen von Stücken: Das 65 Buchstaben lange `RINDFLEISCH...` ist
sofort durch. Mit der Rekursion (und ohne Merken) wären bei langen Wörtern sehr viele Aufteilungen zu prüfen.

#### Aufgabe E.2

Teilaufgabe 2 der Originalaufgabe fragt nach Wörtern, die man **nicht** schreiben kann. Schreibe `unmoegliche(laenge)`: Sie baut alle Wörter
aus den Buchstaben `A` bis `Z` mit dieser Länge (`26 ** laenge` Stück; Hilfe: Man erhält alle Wörter der Länge `k + 1`, indem man an
jedes Wort der Länge `k` jeden Buchstaben anhängt) und gibt die Liste der Wörter zurück, die `zerlege_kennzeichen` ablehnt.

**Erwartete Ausgabe:**
```
2 Buchstaben: 182 von 676 Wörtern gehen nicht, zum Beispiel ['IA', 'IB', 'IC', 'ID', 'IE']
3 Buchstaben: 3978 von 17576 Wörtern gehen nicht, zum Beispiel ['IAA', 'IAB', 'IAC', 'IAD', 'IAE']
```

In [ ]:
BUCHSTABEN = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def unmoegliche(laenge):
    woerter = ['']                       # das leere Wort
    # dein Code: Wörter der Länge 1, 2, ... bis laenge bilden
    ergebnis = []
    # dein Code: nur die behalten, die nicht gehen
    return ergebnis

for laenge in [2, 3]:
    liste = unmoegliche(laenge)
    print(laenge, 'Buchstaben:', len(liste), 'von', 26 ** laenge, 'Wörtern gehen nicht, zum Beispiel', liste[:5])

<details>
<summary>Tipp anzeigen</summary>

Für jede Runde `for _ in range(laenge)`: Bilde eine neue Liste `neu`, indem du an jedes Wort aus `woerter` jeden Buchstaben anhängst.
Danach prüfst du jedes Wort mit `zerlege_kennzeichen(wort) is None`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
BUCHSTABEN = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def unmoegliche(laenge):
    woerter = ['']                       # das leere Wort
    for _ in range(laenge):
        neu = []
        for wort in woerter:
            for buchstabe in BUCHSTABEN:
                neu.append(wort + buchstabe)
        woerter = neu
    ergebnis = []
    for wort in woerter:
        if zerlege_kennzeichen(wort) is None:
            ergebnis.append(wort)
    return ergebnis

for laenge in [2, 3]:
    liste = unmoegliche(laenge)
    print(laenge, 'Buchstaben:', len(liste), 'von', 26 ** laenge, 'Wörtern gehen nicht, zum Beispiel', liste[:5])
```

Es gibt 182 Wörter mit zwei Buchstaben, die nicht gehen, und 3978 mit drei Buchstaben. Ja, es gibt also sogar Wörter mit nur
zwei Buchstaben, die man nicht schreiben kann (zum Beispiel `IA`): Ein Kennzeichen braucht mindestens einen Buchstaben für das Kürzel
und einen für den Mittelteil, und bei zwei Buchstaben muss der **erste** ein Kürzel für sich sein. Bei drei Buchstaben sind es 3978 von 17 576 Wörtern, knapp jedes vierte. Ein Wort mit vier Buchstaben, das nicht geht, ist `TIMO`.

</details>

## Wettbewerbs-Ecke 2: Voll daneben (für Schnelle)

*Angelehnt an Aufgabe 3 „Voll daneben“ der 1. Runde des 37. Bundeswettbewerbs Informatik (2018/19).*

Beim Gewinnspiel von Al setzen die Teilnehmer auf Glückszahlen von 1 bis 1000. Al wählt danach **m Zahlen**. Jeder Teilnehmer bekommt als
Gewinn den **Abstand** seiner Glückszahl zur **nächsten** Zahl von Al, und Al möchte möglichst wenig auszahlen. Wir vereinfachen: Statt
10 Zahlen wählt Al zunächst **zwei**, und wir benutzen nur 12 Glückszahlen (`daten/glueckszahlen.txt`, aus den Lösungshinweisen).

Wir benutzen zwei Beobachtungen (die Lösungsbeschreibung zu 37 begründet sie): Al wählt am besten Zahlen, die Teilnehmer gewählt haben, und die
**sortierten** Glückszahlen zerfallen in **Gruppen benachbarter Zahlen**, wobei jede Gruppe von einer Zahl von Al bedient wird. Am
günstigsten für eine Gruppe ist ihr **Median** (die mittlere Zahl). Die Gruppe der Zahlen `t[i]` bis `t[j]` hat dann diese Kosten:

- Links vom Median `t[mitte]` (Positionen `i` bis `mitte - 1`) kostet jede Zahl `t[mitte] - Zahl`: zusammen `(mitte - i)` mal `t[mitte]`
  minus die **Summe** der Zahlen von `i` bis `mitte - 1`.
- Rechts davon (Positionen `mitte + 1` bis `j`) kostet jede Zahl `Zahl - t[mitte]`: die **Summe** der Zahlen von `mitte + 1` bis `j` minus
  `(j - mitte)` mal `t[mitte]`.

Die beiden Summen sind **Abschnittssummen**, und die Kosten einer Gruppe braucht man **sehr oft**: genau der Fall für Präfixsummen.

#### Aufgabe E.3

Schreibe `kosten_gruppe(t, praefix, i, j)`: Sie gibt die Kosten zurück, wenn die Glückszahlen `t[i]` bis `t[j]` (beide eingeschlossen)
von ihrem **Median** `t[mitte]` mit `mitte = (i + j) // 2` bedient werden. Benutze `praefix` für die beiden Abschnittssummen (mit `abschnitt` wie in 6.1 oder
direkt mit zwei Subtraktionen). Zur Kontrolle rechnet `auszahlung` die Kosten direkt.

**Erwartete Ausgabe:**
```
[7, 10, 14, 16, 18, 19, 20, 24, 26, 29, 30, 32]
Gruppe 2 bis 6: 9 Kontrolle: 9
Gruppe 0 bis 11: 77 Kontrolle: 77
Gruppe 3 bis 3: 0
```

In [ ]:
def lies_zahlen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    return sorted(zahlen)

def baue_praefix(zahlen):
    praefix = [0]
    for zahl in zahlen:
        praefix.append(praefix[-1] + zahl)
    return praefix

def abstand_zur_naechsten(zahl, al):
    beste = abs(zahl - al[0])
    for a in al:
        if abs(zahl - a) < beste:
            beste = abs(zahl - a)
    return beste

def auszahlung(t, al):
    """Direkte Rechnung: Summe der Abstände aller Glückszahlen zur nächsten Zahl von Al."""
    summe = 0
    for zahl in t:
        summe = summe + abstand_zur_naechsten(zahl, al)
    return summe

T = lies_zahlen('daten/glueckszahlen.txt')
PRAEFIX = baue_praefix(T)

def kosten_gruppe(t, praefix, i, j):
    mitte = (i + j) // 2
    return 0                             # ersetze diese Zeile

print(T)
print('Gruppe 2 bis 6:', kosten_gruppe(T, PRAEFIX, 2, 6), 'Kontrolle:', auszahlung(T[2:7], [T[4]]))
print('Gruppe 0 bis 11:', kosten_gruppe(T, PRAEFIX, 0, 11), 'Kontrolle:', auszahlung(T, [T[5]]))
print('Gruppe 3 bis 3:', kosten_gruppe(T, PRAEFIX, 3, 3))

<details>
<summary>Tipp anzeigen</summary>

Summe von `t[i]` bis `t[mitte - 1]`: `praefix[mitte] - praefix[i]`. Summe von `t[mitte + 1]` bis `t[j]`: `praefix[j + 1] - praefix[mitte + 1]`. Die
Kosten sind `links + rechts` wie oben beschrieben.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_zahlen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    return sorted(zahlen)

def baue_praefix(zahlen):
    praefix = [0]
    for zahl in zahlen:
        praefix.append(praefix[-1] + zahl)
    return praefix

def abstand_zur_naechsten(zahl, al):
    beste = abs(zahl - al[0])
    for a in al:
        if abs(zahl - a) < beste:
            beste = abs(zahl - a)
    return beste

def auszahlung(t, al):
    """Direkte Rechnung: Summe der Abstände aller Glückszahlen zur nächsten Zahl von Al."""
    summe = 0
    for zahl in t:
        summe = summe + abstand_zur_naechsten(zahl, al)
    return summe

T = lies_zahlen('daten/glueckszahlen.txt')
PRAEFIX = baue_praefix(T)

def kosten_gruppe(t, praefix, i, j):
    mitte = (i + j) // 2
    links = t[mitte] * (mitte - i) - (praefix[mitte] - praefix[i])               # Summe von t[i] bis t[mitte - 1]
    rechts = (praefix[j + 1] - praefix[mitte + 1]) - t[mitte] * (j - mitte)       # Summe von t[mitte + 1] bis t[j]
    return links + rechts

print(T)
print('Gruppe 2 bis 6:', kosten_gruppe(T, PRAEFIX, 2, 6), 'Kontrolle:', auszahlung(T[2:7], [T[4]]))
print('Gruppe 0 bis 11:', kosten_gruppe(T, PRAEFIX, 0, 11), 'Kontrolle:', auszahlung(T, [T[5]]))
print('Gruppe 3 bis 3:', kosten_gruppe(T, PRAEFIX, 3, 3))
```

Für die Gruppe 14, 16, 18, 19, 20 (Positionen 2 bis 6, Median 18) sind es `4 + 2 + 0 + 1 + 2 = 9`. Eine Gruppe aus einer einzigen Zahl kostet 0. Jede Gruppe
kostet jetzt **zwei Subtraktionen** und ein paar Multiplikationen, ganz gleich wie viele Zahlen sie hat. Bei der direkten Rechnung
wären es so viele Rechenschritte wie Zahlen in der Gruppe.

</details>

#### Aufgabe E.4

Teile die sortierten Glückszahlen in **zwei** Gruppen: `t[0..schnitt-1]` und `t[schnitt..n-1]` mit `schnitt` von 1 bis `n - 1`. Schreibe `beste_zwei(t, praefix)`:
Sie probiert alle `schnitt` und gibt `(kleinste Kosten, schnitt)` zurück. Die Kosten einer Teilung sind die Kosten der linken plus die Kosten der
rechten Gruppe, jeweils mit `kosten_gruppe` berechnet.

**Erwartete Ausgabe:**
```
Schnitt vor Position 7 , Al wählt [16, 29] , Auszahlung 38 , Kontrolle 38
```

In [ ]:
def lies_zahlen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    return sorted(zahlen)

def baue_praefix(zahlen):
    praefix = [0]
    for zahl in zahlen:
        praefix.append(praefix[-1] + zahl)
    return praefix

def abstand_zur_naechsten(zahl, al):
    beste = abs(zahl - al[0])
    for a in al:
        if abs(zahl - a) < beste:
            beste = abs(zahl - a)
    return beste

def auszahlung(t, al):
    """Direkte Rechnung: Summe der Abstände aller Glückszahlen zur nächsten Zahl von Al."""
    summe = 0
    for zahl in t:
        summe = summe + abstand_zur_naechsten(zahl, al)
    return summe

T = lies_zahlen('daten/glueckszahlen.txt')
PRAEFIX = baue_praefix(T)

def kosten_gruppe(t, praefix, i, j):
    mitte = (i + j) // 2
    links = t[mitte] * (mitte - i) - (praefix[mitte] - praefix[i])
    rechts = (praefix[j + 1] - praefix[mitte + 1]) - t[mitte] * (j - mitte)
    return links + rechts

def beste_zwei(t, praefix):
    n = len(t)
    return 0, 1                          # ersetze diese Zeile

kosten, schnitt = beste_zwei(T, PRAEFIX)
al = [T[(0 + schnitt - 1) // 2], T[(schnitt + len(T) - 1) // 2]]
print('Schnitt vor Position', schnitt, ', Al wählt', al, ', Auszahlung', kosten, ', Kontrolle', auszahlung(T, al))

<details>
<summary>Tipp anzeigen</summary>

Eine Schleife `for schnitt in range(1, n)`. In jeder Runde rufst du zweimal `kosten_gruppe` auf: für `0` bis `schnitt - 1` und für `schnitt` bis `n - 1`.
Merke dir das Minimum und den zugehörigen `schnitt`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_zahlen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    return sorted(zahlen)

def baue_praefix(zahlen):
    praefix = [0]
    for zahl in zahlen:
        praefix.append(praefix[-1] + zahl)
    return praefix

def abstand_zur_naechsten(zahl, al):
    beste = abs(zahl - al[0])
    for a in al:
        if abs(zahl - a) < beste:
            beste = abs(zahl - a)
    return beste

def auszahlung(t, al):
    """Direkte Rechnung: Summe der Abstände aller Glückszahlen zur nächsten Zahl von Al."""
    summe = 0
    for zahl in t:
        summe = summe + abstand_zur_naechsten(zahl, al)
    return summe

T = lies_zahlen('daten/glueckszahlen.txt')
PRAEFIX = baue_praefix(T)

def kosten_gruppe(t, praefix, i, j):
    mitte = (i + j) // 2
    links = t[mitte] * (mitte - i) - (praefix[mitte] - praefix[i])
    rechts = (praefix[j + 1] - praefix[mitte + 1]) - t[mitte] * (j - mitte)
    return links + rechts

def beste_zwei(t, praefix):
    n = len(t)
    beste_kosten = None
    bester_schnitt = None
    for schnitt in range(1, n):
        kosten = kosten_gruppe(t, praefix, 0, schnitt - 1) + kosten_gruppe(t, praefix, schnitt, n - 1)
        if beste_kosten is None or kosten < beste_kosten:
            beste_kosten = kosten
            bester_schnitt = schnitt
    return beste_kosten, bester_schnitt

kosten, schnitt = beste_zwei(T, PRAEFIX)
al = [T[(0 + schnitt - 1) // 2], T[(schnitt + len(T) - 1) // 2]]
print('Schnitt vor Position', schnitt, ', Al wählt', al, ', Auszahlung', kosten, ', Kontrolle', auszahlung(T, al))
```

Eine Probe: Der Code in der letzten Zeile berechnet die Median-Zahlen der beiden Gruppen und lässt die direkte Funktion `auszahlung` die
Kosten nachrechnen; sie stimmen überein. Es werden nur `n - 1 = 11` Teilungen probiert, jede mit zwei Gruppen in **O(1)**.

</details>

#### Aufgabe E.5

Jetzt **m** Zahlen für Al. Die Tabelle `best[l][j]` ist die kleinste Auszahlung für die ersten `j` Glückszahlen (`t[0..j-1]`), wenn Al dafür genau
`l` Zahlen wählt, also die ersten `j` Zahlen in `l` Gruppen geteilt werden. Für `l = 0` geht es nur mit `j = 0` (Kosten 0), alles andere ist
unmöglich (`UNENDLICH`). Für `l >= 1` beginnt die **letzte** Gruppe an einer Stelle `i` (mit `l - 1 <= i < j`), sie umfasst `t[i..j-1]` und
kostet `kosten_gruppe(t, praefix, i, j - 1)`. Davor liegen die ersten `i` Zahlen in `l - 1` Gruppen:

`best[l][j] = Minimum über i von ( best[l - 1][i] + kosten_gruppe(t, praefix, i, j - 1) )`

Fülle die Tabelle von klein nach groß (`l` aufsteigend, `j` aufsteigend, `j >= l`) und gib sie zurück. Die Lösungshinweise geben für diese 12
Zahlen und 5 Zahlen von Al die Auszahlung 12 an.

**Erwartete Ausgabe:**
```
1 Zahlen für Al: Auszahlung 77
2 Zahlen für Al: Auszahlung 38
3 Zahlen für Al: Auszahlung 24
4 Zahlen für Al: Auszahlung 17
5 Zahlen für Al: Auszahlung 12
```

In [ ]:
def lies_zahlen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    return sorted(zahlen)

def baue_praefix(zahlen):
    praefix = [0]
    for zahl in zahlen:
        praefix.append(praefix[-1] + zahl)
    return praefix

def abstand_zur_naechsten(zahl, al):
    beste = abs(zahl - al[0])
    for a in al:
        if abs(zahl - a) < beste:
            beste = abs(zahl - a)
    return beste

def auszahlung(t, al):
    """Direkte Rechnung: Summe der Abstände aller Glückszahlen zur nächsten Zahl von Al."""
    summe = 0
    for zahl in t:
        summe = summe + abstand_zur_naechsten(zahl, al)
    return summe

T = lies_zahlen('daten/glueckszahlen.txt')
PRAEFIX = baue_praefix(T)

def kosten_gruppe(t, praefix, i, j):
    mitte = (i + j) // 2
    links = t[mitte] * (mitte - i) - (praefix[mitte] - praefix[i])
    rechts = (praefix[j + 1] - praefix[mitte + 1]) - t[mitte] * (j - mitte)
    return links + rechts

UNENDLICH = 10 ** 9

def beste_tabelle(t, praefix, m):
    n = len(t)
    best = [[UNENDLICH] * (n + 1) for l in range(m + 1)]
    best[0][0] = 0
    # dein Code
    return best

best = beste_tabelle(T, PRAEFIX, 5)
for l in range(1, 6):
    print(l, 'Zahlen für Al: Auszahlung', best[l][len(T)])

<details>
<summary>Tipp anzeigen</summary>

Drei verschachtelte Schleifen: `for l in range(1, m + 1)`, `for j in range(l, n + 1)` und `for i in range(l - 1, j)`. In der innersten
rechnest du `best[l - 1][i] + kosten_gruppe(t, praefix, i, j - 1)` und behältst das Minimum in `best[l][j]`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_zahlen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    return sorted(zahlen)

def baue_praefix(zahlen):
    praefix = [0]
    for zahl in zahlen:
        praefix.append(praefix[-1] + zahl)
    return praefix

def abstand_zur_naechsten(zahl, al):
    beste = abs(zahl - al[0])
    for a in al:
        if abs(zahl - a) < beste:
            beste = abs(zahl - a)
    return beste

def auszahlung(t, al):
    """Direkte Rechnung: Summe der Abstände aller Glückszahlen zur nächsten Zahl von Al."""
    summe = 0
    for zahl in t:
        summe = summe + abstand_zur_naechsten(zahl, al)
    return summe

T = lies_zahlen('daten/glueckszahlen.txt')
PRAEFIX = baue_praefix(T)

def kosten_gruppe(t, praefix, i, j):
    mitte = (i + j) // 2
    links = t[mitte] * (mitte - i) - (praefix[mitte] - praefix[i])
    rechts = (praefix[j + 1] - praefix[mitte + 1]) - t[mitte] * (j - mitte)
    return links + rechts

UNENDLICH = 10 ** 9

def beste_tabelle(t, praefix, m):
    n = len(t)
    best = [[UNENDLICH] * (n + 1) for l in range(m + 1)]
    best[0][0] = 0
    for l in range(1, m + 1):            # erst alle Zeilen mit weniger Zahlen für Al ...
        for j in range(l, n + 1):        # ... in jeder Zeile von klein nach groß
            for i in range(l - 1, j):    # die letzte Gruppe t[i..j-1] beginnt bei i
                kosten = best[l - 1][i] + kosten_gruppe(t, praefix, i, j - 1)
                if kosten < best[l][j]:
                    best[l][j] = kosten
    return best

best = beste_tabelle(T, PRAEFIX, 5)
for l in range(1, 6):
    print(l, 'Zahlen für Al: Auszahlung', best[l][len(T)])
```

Die Auszahlung sinkt mit jeder zusätzlichen Zahl für Al; bei `l = 5` sind es 12, wie in den Lösungshinweisen. Bei `l = 2` ist es dasselbe wie das Ergebnis aus
E.4. Die Tabelle hat `m · n` Einträge, jeder probiert bis zu `n` Anfänge der letzten Gruppe, jeder Versuch kostet dank der Präfixsummen **O(1)**:
Laufzeit **O(m · n²)**. Ohne Präfixsummen käme noch ein Faktor `n` dazu. Das ist das Verfahren aus dem
Lösungsnotebook zu Voll daneben, das die vollen Daten mit 100 bis 199 Teilnehmern und 10 Zahlen löst.

</details>

## Das hast du gelernt

```python
# Tabelle von klein nach groß: tabelle[i] aus kleineren Einträgen, ohne Rekursion
tabelle = [0] * (n + 1)
tabelle[0] = 1                           # Rand zuerst
for i in range(1, n + 1):                # Reihenfolge: alles, was i braucht, ist schon fertig
    ...                                  # tabelle[i] aus tabelle[i - 1], tabelle[j] mit j < i, ...

# Zurückverfolgen: je Eintrag nur die letzte Entscheidung merken
wahl[i] = ...                            # in der Schleife, wenn ein Eintrag besser wird
while i > 0:
    liste.append(wahl[i]); i = i - ...   # dann von hinten nach vorn abarbeiten

# Präfixsummen: Summe der ersten i Zahlen (praefix[0] = 0)
praefix = [0]
for zahl in zahlen:
    praefix.append(praefix[-1] + zahl)
summe = praefix[b + 1] - praefix[a]      # Zahlen von Position a bis b, beide eingeschlossen
```

- **Von klein nach groß**: Die Formeln sind dieselben wie in Notebook 10, aber **du** wählst die Reihenfolge der Berechnung. Das Wichtigste: Wenn
  ein Eintrag berechnet wird, müssen alle, die er braucht, **schon fertig** sein. Sonst liest man den Anfangswert (`None`, `0`) und Python meldet **keinen** Fehler.
- **Vorteile gegenüber dem Wörterbuch**: keine Rekursionstiefe, keine Funktionsaufrufe, und eine Tabelle beantwortet gleich alle kleineren Fragen.
  Nachteil: Man rechnet alle Einträge, auch die, die man nicht braucht.
- **Laufzeit** = Zahl der Tabelleneinträge × Arbeit je Eintrag (Gitter: z · s, Münzen: Betrag · Sorten, Wort in Wörter: n · L).
  Speicher = Größe der Tabelle; oft braucht man nur die letzte Zeile.
- **Wort in Wörter zerlegen**: `geht[i]` mit `geht[0] = True`; `geht[i]` ist wahr, wenn es ein `j < i` mit `geht[j]` und `wort[j:i]` erlaubt gibt. Mit `von[i]`
  zurückverfolgen. Nicht gierig das längste Wort nehmen.
- **Präfixsummen**: einmal `n` Additionen, danach jede Abschnittssumme in **O(1)**. Die führende `0` vermeidet den Sonderfall `a = 0`; prüfe
  Formeln am Rand und achte auf die Verschiebung um eins.
- Typische Fehler: falsche Reihenfolge, negative Indizes in Python (zählen von hinten), Verschiebung um eins.

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Auto-Scrabble](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/36_AutoScrabble/autoscrabble_loesung.ipynb) (36. BwInf): Wort in Kennzeichen zerlegen, Tabelle `geht[i]`
- [Nummernmerker](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/38_Nummernmerker/nummernmerker_loesung.ipynb) (38.): Nummern in Blöcke teilen, Tabelle von vorne nach hinten
- [Voll daneben](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/37_VollDaneben/voll_daneben_loesung.ipynb) (37.): Gruppen mit Präfixsummen
- [Flaschenzug](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/34_Flaschenzug/flaschenzug_loesung.ipynb) (34.): Tabelle und Präfixsummen
- [Zaras dritter Fehler](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/30_ZarasDritterFehler/zarasdritterfehler_loesung.ipynb) (30.): Präfixsummen

Im nächsten Notebook (12) geht es um **gierige Verfahren**: Man trifft immer die scheinbar beste Entscheidung und rechnet nicht alles durch. Das ist oft
viel schneller, aber nicht immer richtig. Du lernst, wann es stimmt und wie man es **begründet**.